# EdgeInspect-AI - GPU training on Kaggle

Settings: **Accelerator = GPU** (T4 or P100), **Internet = On**. Add the uploaded dataset `edgeinspect-neu-det` as input.

Change `EXP_ID` / `IMGSZ` below for each experiment. Run all cells top to bottom.

In [ ]:
EXP_ID = 'EXP-001'
IMGSZ = 640
EPOCHS = 100
BATCH = 16

In [ ]:
# 1) Same Ultralytics version as the laptop -> same training code
!pip install -q ultralytics==8.4.163
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch, ultralytics; print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), '| ultralytics', ultralytics.__version__)

In [ ]:
# 2) Copy the project from the read-only input folder to a writable one (/tmp), and fix the dataset path
import glob, os, shutil, re
found = [d for d in glob.glob('/kaggle/input/**/EdgeInspect-AI', recursive=True) if os.path.isdir(d)]
if not found:  # Kaggle left the upload zipped -> extract it ourselves
    import zipfile
    zips = glob.glob('/kaggle/input/**/*.zip', recursive=True)
    assert zips, 'No EdgeInspect-AI folder or zip under /kaggle/input: ' + str(glob.glob('/kaggle/input/**', recursive=True)[:20])
    zipfile.ZipFile(zips[0]).extractall('/kaggle/tmp_extract')
    found = ['/kaggle/tmp_extract/EdgeInspect-AI']
src = found[0]
print('project source:', src)
dst = '/tmp/EdgeInspect-AI'  # not /kaggle/working, so the Output tab holds only EXP-xxx.zip
if not os.path.exists(dst):
    shutil.copytree(src, dst)
os.chdir(dst)
yaml_path = 'training/neu_det.yaml'
txt = open(yaml_path).read()
txt = re.sub(r'^path: .*$', f'path: {dst}/dataset/processed/neu_det', txt, flags=re.M)
open(yaml_path, 'w').write(txt)
print(txt)
for sp in ['train', 'val', 'test']:
    print(sp, len(os.listdir(f'dataset/processed/neu_det/images/{sp}')), 'images')

In [ ]:
# 3) Train (same script and settings as on the laptop, only device=GPU)
!python training/train.py --name {EXP_ID} --imgsz {IMGSZ} --epochs {EPOCHS} --batch {BATCH} --device 0 --workers 4

In [ ]:
# 4) Package the run for download (Output tab -> EXP-xxx.zip)
import subprocess, json
info = {'gpu': subprocess.getoutput('nvidia-smi --query-gpu=name --format=csv,noheader'),
        'torch': torch.__version__, 'ultralytics': ultralytics.__version__}
json.dump(info, open(f'experiments/{EXP_ID}/train_env.json', 'w'), indent=2)
shutil.make_archive(f'/kaggle/working/{EXP_ID}', 'zip', 'experiments', EXP_ID)
print('Download /kaggle/working/' + EXP_ID + '.zip from the Output panel')